<div style="text-align: justify; max-width: 90ch">

# Lecture 03a: Tracing and deterministic scorers

**Status: Mandatory reading.**

Lecture 2 built a RAG system and measured its retrieval: in `lec_02d` the right file was among the four retrieved chunks for almost every question of the evaluation set.
But is the *answer* right, does it name its source, does it say so when the notes are silent? This notebook first records what every step did for one answer, with **MLflow tracing**;
then it turns grading into numbers, with [MLflow's evaluation](https://mlflow.org/docs/latest/genai/eval-monitor/) and checks written in Python.

**What this notebook covers**

- Why one good answer proves nothing.
- **Tracing**: what each step of one answer received and returned, in the MLflow UI and from code.
- The evaluation set as MLflow records.
- **Scorers** written with `@scorer`, run by `mlflow.genai.evaluate`.
- Aggregates versus per-question rows.

</div>

In [1]:
import json
import os
import re
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from mlflow.entities import SpanType
from mlflow.genai.scorers import scorer

from course_helpers.checks import check_mlflow
from course_helpers.corpus import load_corpus, load_eval_set
from course_helpers.rag import (
    LECTURE_KINDS,
    build_chroma_index,
    make_rag_chain,
    split_documents,
)

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-03-eval"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from course_helpers.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs.** The system of `lec_02d`, from the `course_helpers` helpers:

- all thirteen lectures in chunks of 800 characters;
- the Chroma index, reused if `lec_02d` built it (otherwise built now, which takes minutes);
- four chunks per question, from the lecture notes and guides (the filter of `lec_02d`), and the same chat model settings.

Then the evaluation set.

</div>

In [3]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

chunks = split_documents(load_corpus(REPO / "corpus" / "uoa_py_course"))
store = build_chroma_index(
    chunks,
    "course_notes",
    DATA_DIR / "chroma",
    embed_model=EMBED_MODEL,
    ollama_host=OLLAMA_HOST,
)
retriever = store.as_retriever(
    search_kwargs={"k": 4, "filter": {"kind": {"$in": LECTURE_KINDS}}}
)

llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=200,
    num_ctx=8192,
)
rag_chain = make_rag_chain(retriever, llm)

eval_set = load_eval_set(REPO / "corpus" / "eval" / "qa_eval_set.jsonl")

print(f"{len(eval_set)} questions in the evaluation set")

Ollama OK at http://localhost:11434; models ready: nomic-embed-text


Reusing 'course_notes': 2452 chunks in chroma/
19 questions in the evaluation set


<div style="text-align: justify; max-width: 90ch">

## 1. One good answer proves nothing

Ask the system one question of the evaluation set, q19: what is the difference between a list and a tuple?

</div>

In [4]:
q19 = next(q for q in eval_set if q["id"] == "q19")

print(rag_chain.invoke(q19["question"]))

The difference between a list and a tuple in Python is that a list is **mutable** (can be modified after creation), while a tuple is **immutable** (cannot be modified after creation).

Source file: `lecture_04/lec_04b_index_slice_list_tuple_set.md`


<div style="text-align: justify; max-width: 90ch">

Is it right? The evaluation set expects two facts (tuples are immutable, lists are mutable) from `lec_04b`. Even if the answer states both, it proves little:

- **One question** says nothing about the others, or about the next question a student asks.
- **Every change** (prompt, `k`, model, chunk size) can fix one answer and break another, and nobody re-reads every answer after every change.
- **Answers vary**: even at temperature 0 the wording can change from run to run.

</div>

<div style="text-align: justify; max-width: 90ch">

The fix is the one the Python course used for models: a test set and a metric computed by code.

| Term | Meaning |
| --- | --- |
| **evaluation set** (from `lec_02d`) | questions with known good answers (`corpus/eval/qa_eval_set.jsonl`) |
| **scorer** | a function that grades one answer and returns a number, or True/False |
| **deterministic scorer** | a scorer written as plain code: the same answer always gets the same score |
| **evaluation run** | an MLflow run holding every answer, its trace and its scores |

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Tracing: what each step received

The answer above came from three steps: the retriever, the prompt and the model. In `lec_02b` you printed each step's output to find the failing one;
for 19 questions, many runs and changing code, that record must be kept automatically. MLflow, which recorded *runs* of a training job in lecture 13 of the Python course,
records **traces** of an application built of steps ([MLflow tracing](https://mlflow.org/docs/latest/genai/tracing/)):

| Term | Meaning |
| --- | --- |
| **trace** | one end-to-end call of your application, for example one `rag_chain.invoke(...)` |
| **span** | one step inside it, with its inputs, outputs and duration; spans nest inside each other |
| **span type** | the kind of step: `RETRIEVER`, `CHAT_MODEL`, `CHAIN` (a pipeline of steps) |

Start the tracking server as in guide `03a_mlflow_server` (in this lecture's `infra_tools/`). One line, [`mlflow.langchain.autolog()`](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langchain/),
makes LangChain report every chain's steps from here on:

</div>

In [5]:
check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

answer_q19 = rag_chain.invoke(q19["question"])  # the question of section 1, now traced

MLflow OK at http://127.0.0.1:5010


In [6]:
trace = mlflow.get_trace(mlflow.get_last_active_trace_id(), flush=True)

pd.DataFrame(
    [
        {
            "span": s.name,
            "type": s.span_type,
            "seconds": round((s.end_time_ns - s.start_time_ns) / 1e9, 2),
        }
        for s in trace.data.spans
    ]
)

,span,type,seconds
0,RunnableSequence,CHAIN,0.59
1,"RunnableParallel<context,question>",CHAIN,0.02
2,RunnableSequence,CHAIN,0.02
3,VectorStoreRetriever,RETRIEVER,0.02
4,RunnablePassthrough,CHAIN,0.00
5,format_context_from,CHAIN,0.00
6,ChatPromptTemplate,CHAIN,0.00
7,ChatOllama,CHAT_MODEL,0.57
8,StrOutputParser,CHAIN,0.00


<div style="text-align: justify; max-width: 90ch">

Each row is one step: LangChain's pieces as `CHAIN` spans, the search as the `RETRIEVER` span, the model as the `CHAT_MODEL` span.

### ⏸ Pause point

- **Open** <http://127.0.0.1:5010>, experiment `llm-course-03-eval`, **Traces** tab, and click the newest trace.
- **Click `VectorStoreRetriever`**: its outputs are the four chunks the model was given. Is one of them from `lec_04b`?
- **Then click `ChatOllama`**: its input is the prompt the model *actually* received, chunks included.

If the answer were wrong, which step would you fix? The same spans, read from code:

</div>

In [7]:
retriever_span = trace.search_spans(span_type=SpanType.RETRIEVER)[0]
sources = [doc["metadata"]["source"] for doc in retriever_span.outputs]

print(f"sources: {sources}")
print(f"tokens: {trace.info.token_usage}")

sources: ['lecture_04/lec_04b_index_slice_list_tuple_set.md', 'lecture_05/practice_lec_05a_boolean.md', 'lecture_04/lec_04b_index_slice_list_tuple_set.md', 'lecture_04/index_slice_list_exercise.md']
tokens: {'input_tokens': 759, 'output_tokens': 61, 'total_tokens': 820}


<div style="text-align: justify; max-width: 90ch">

`search_spans` finds spans by type, which is how a scorer will read a trace (section 4). Compare the sources with the file the answer cites.
The **token usage** is the number a hosted provider would bill: the prompt with its four chunks, plus the answer.

Autolog covers LangChain. For your own functions, a **decorator** does the same: a line starting with `@` just above a `def`, which wraps the function so that every call goes through the wrapper first.
`@mlflow.trace` is such a wrapper: it records each call as a span, with the arguments as inputs and the return value as outputs ([manual tracing](https://mlflow.org/docs/latest/genai/tracing/app-instrumentation/manual-tracing/)):

</div>

In [8]:
@mlflow.trace(span_type=SpanType.CHAIN)
def answer_with_sources_for(question):
    """Answer with the RAG chain, and list the files the retriever returns."""
    sources = [doc.metadata["source"] for doc in retriever.invoke(question)]
    return {"answer": rag_chain.invoke(question), "sources": sources}


result = answer_with_sources_for(q19["question"])
trace = mlflow.get_trace(mlflow.get_last_active_trace_id(), flush=True)
span_names = [s.name for s in trace.data.spans]

print(f"{len(span_names)} spans, the first four: {span_names[:4]}")

11 spans, the first four: ['answer_with_sources_for', 'VectorStoreRetriever', 'RunnableSequence', 'RunnableParallel<context,question>']


<div style="text-align: justify; max-width: 90ch">

The decorated function is the outer span, and the retriever call and the chain nest inside it: one trace for the whole call.
The decorator `@scorer` of section 4 wraps a function too.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. The evaluation set as MLflow records

`mlflow.genai.evaluate` reads a list of **records** ([evaluation datasets](https://mlflow.org/docs/latest/genai/datasets/)),
each a dict with two keys:

- **`inputs`**: the keyword arguments of the system under test, here `question`.
- **`expectations`**: what a good answer contains; scorers read it, the system never sees it.

A demonstration needs few questions: four with short answers from the set, and one the notes cannot answer, which has no expectations.
The full set of 19 is an exercise: Exercise 3.1.

</div>

In [9]:
def build_eval_records_from(eval_set):
    """Turn evaluation-set rows into MLflow records: inputs and expectations."""
    return [
        {
            "inputs": {"question": row["question"]},
            "expectations": {
                "expected_source": row["expected_source"],
                **row["expectations"],  # adds expected_facts
            },
        }
        for row in eval_set
    ]


DEMO_IDS = ["q13", "q15", "q16", "q19"]

records = build_eval_records_from([q for q in eval_set if q["id"] in DEMO_IDS])
records.append(
    {"inputs": {"question": "Which lecture covers regular expressions?"}}
)  # not in the notes
q19_record = records[DEMO_IDS.index("q19")]

print(json.dumps(q19_record, indent=2))

{
  "inputs": {
    "question": "What is the difference between a list and a tuple?"
  },
  "expectations": {
    "expected_source": "lecture_04/lec_04b_index_slice_list_tuple_set.md",
    "expected_facts": [
      "Tuples are immutable.",
      "Lists are mutable."
    ]
  }
}


<div style="text-align: justify; max-width: 90ch">

The q19 record keeps the question apart from what we check: the expected file and the expected facts.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Scorers

A scorer receives what it needs by parameter name and returns its verdict. `@scorer` ([code-based scorers](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/custom/)) is a decorator,
like `@mlflow.trace` in section 2: it turns the function into one MLflow can call.

| Parameter | MLflow fills in |
| --- | --- |
| `inputs` | the record's inputs (the question) |
| `outputs` | what the system returned (the answer) |
| `expectations` | the record's expectations |
| `trace` | the trace of the call that produced the answer |

The first check: does the answer name the expected file, as the prompt of `lec_02d` asks? A scorer that returns `None` abstains:
for the question without an expected file, there is nothing to check.

</div>

In [10]:
@scorer
def cites_expected_source(outputs, expectations):
    """Does the answer name the expected file (its name without .md)?"""
    if not expectations:
        return None  # no expected file: nothing to check

    return Path(expectations["expected_source"]).stem in outputs

<div style="text-align: justify; max-width: 90ch">

A scorer is still a function: call it on answers you wrote yourself, where you know the right verdict.

</div>

In [11]:
citations = [
    "Source file: `lec_04b_index_slice_list_tuple_set.md`",
    "This is covered in lecture 4b, on lists and tuples.",
]

for citation in citations:
    verdict = cites_expected_source(
        outputs=citation, expectations=q19_record["expectations"]
    )
    print(f"{verdict} | {citation}")

True | Source file: `lec_04b_index_slice_list_tuple_set.md`
False | This is covered in lecture 4b, on lists and tuples.


<div style="text-align: justify; max-width: 90ch">

The second citation fails on every run: it names the right lecture in words, and the scorer looks only for the file name.
A deterministic scorer is strict and blind to meaning, so you decide what counts and write it down.

</div>

<div style="text-align: justify; max-width: 90ch">

Three more checks, each one idea:

- **`retrieval_hit`** reads the trace: was a chunk of the expected file among the chunks the model was given? The RETRIEVER span of section 2, found with `trace.search_spans`.
- **`refused`** looks for phrases of a refusal, collected from real answers. In example runs models seldom repeated the prompt's own words, so the list also holds the phrasings they used.
- **`fact_keywords_found`** is a crude first check of content: a fact counts as stated when at least half of its keywords (common words dropped) appear in the answer.

</div>

In [12]:
@scorer
def retrieval_hit(trace, expectations):
    """Was a chunk of the expected file among the chunks this answer was given?"""
    if not expectations:
        return None

    retriever_span = trace.search_spans(span_type=SpanType.RETRIEVER)[0]
    sources = [doc["metadata"]["source"] for doc in retriever_span.outputs]
    return expectations["expected_source"] in sources

In [13]:
REFUSAL_PHRASES = [
    "not contain",
    "not mention",
    "not cover",
    "no information",
    "not explicitly mentioned",
    "cannot determine",
    "cannot answer",
]


@scorer
def refused(outputs):
    """Does the answer say that the notes do not contain the answer?"""
    text = outputs.lower().replace("n't", " not")  # "don't" -> "do not"
    return any(phrase in text for phrase in REFUSAL_PHRASES)

In [14]:
STOPWORDS = set(
    "a an and are as at be by can do does for from has have how in into is it its "
    "of on or that the their them they this to use uses was when which with you your".split()
)


def extract_keywords_from(fact):
    """The words of a fact that carry meaning: lower case, common words dropped."""
    words = re.findall(r"[a-z0-9_]+", fact.lower())  # runs of letters and digits
    return [w for w in words if len(w) > 1 and w not in STOPWORDS]


@scorer
def fact_keywords_found(outputs, expectations):
    """Share of the expected facts with at least half of their keywords in the answer."""
    if not expectations:
        return None

    text = outputs.lower()
    facts = expectations["expected_facts"]
    found = 0

    for fact in facts:
        keywords = extract_keywords_from(fact)
        if sum(k in text for k in keywords) >= len(keywords) / 2:
            found += 1

    return found / len(facts)

<div style="text-align: justify; max-width: 90ch">

## 5. Running an evaluation

`mlflow.genai.evaluate(data=..., predict_fn=..., scorers=[...])`:

- calls `predict_fn` once per record, with the record's `inputs` as keyword arguments, so its parameter must be called `question`;
- traces each call, and passes each answer, with its record and trace, to every scorer;
- logs everything to one **evaluation run** in the current experiment.

MLflow first tries one question as a test (an INFO line says so), then runs all of them, and ends with a button that opens the results.

</div>

In [15]:
def predict(question):
    """The system under test: the RAG chain of lec_02d."""
    return rag_chain.invoke(question)


SCORERS = [cites_expected_source, retrieval_hit, refused, fact_keywords_found]

results = mlflow.genai.evaluate(data=records, predict_fn=predict, scorers=SCORERS)

2026/10/12 19:05:31 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


2026/10/12 19:05:31 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


In [16]:
pd.Series(results.metrics).sort_index().round(2)

cites_expected_source/mean    0.75
fact_keywords_found/mean      0.88
refused/mean                  0.20
retrieval_hit/mean            1.00
dtype: float64

<div style="text-align: justify; max-width: 90ch">

Each line is one number for the whole system: the mean of a True/False scorer is the share of answers that pass, and an abstaining `None` is left out of the mean.
Is the value of `refused/mean` good or bad? The aggregate cannot say.

### ⏸ Pause point

Click **View evaluation results in MLflow** above (or open experiment `llm-course-03-eval` and its newest run). There is one row per question with its answer and the scorers' verdicts;
a row opens its trace.

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. Aggregates hide the questions

A mean says how often, not which. The same results, one row per question:

</div>

In [17]:
frame = results.result_df
# "request" holds the question as text, or as {"question": ...}, depending on the trace
questions = [r["question"] if isinstance(r, dict) else r for r in frame["request"]]

rows = pd.DataFrame(
    {s.name: frame[f"{s.name}/value"].to_numpy() for s in SCORERS},
    index=[question[:45] for question in questions],
)

rows

,cites_expected_source,retrieval_hit,refused,fact_keywords_found
What is the naive assumption in Naive Bayes?,True,True,False,0.5
Where can I deploy a Gradio app so that anyon,True,True,False,1.0
What is the difference between loc and iloc i,False,True,False,1.0
What is the difference between a list and a t,True,True,False,1.0
Which lecture covers regular expressions?,NaN,NaN,True,NaN


<div style="text-align: justify; max-width: 90ch">

Find the row of the question on regular expressions: a refusal there is the right behaviour, and on the other rows it would be a failure.
A real evaluation set mixes both kinds of question, with an expectation that says which behaviour is right. Read the other rows too:
where an answer fails a check, open its trace in the UI and decide whether the answer is wrong or only the check.

One caution: the same 19 questions chose `k=4` and the filter in `lec_02d`, so their scores are optimistic (the leakage of `lec_13b` in the Python course);
the optional `lec_03f` splits them into questions for tuning and questions for grading.

### ⏸ Pause point

Which of the four scorers would you trust to block a release, and which only to point you at answers worth reading?

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- One good answer proves nothing: an **evaluation set** and **scorers** turn "it seems to work" into numbers anyone can recompute.
- A **trace** records every step of one call as **spans**, with their inputs, outputs and type. `mlflow.langchain.autolog()` traces LangChain;
  the decorator `@mlflow.trace` traces your own functions.
- MLflow **records** hold `inputs` and `expectations`. A **scorer** is a function with `@scorer` that receives `inputs`,
  `outputs`, `expectations` or `trace` by name, and may abstain with `None`.
- `mlflow.genai.evaluate` runs the system on every record and logs an **evaluation run**.
- Read the **rows**, not only the means: the same `refused` value is right on one question and wrong on another.
- **Deterministic scorers** are exact but blind to meaning: `cites_expected_source` fails a citation written in words.

</div>

<div style="text-align: justify; max-width: 90ch">

The helpers live in `course_helpers.evaluation`: `build_eval_records_from`, the scorers, `per_question_table` (section 6), and `DETERMINISTIC_SCORERS`:
the four scorers plus `answer_tokens`, the output tokens from the trace.

**Next: `lec_03b`**, an LLM judge that reads whether the answer states the expected facts, and two variants compared on the same questions.

</div>